# Credit Card Fraud Detection — XGBoost + SMOTE

Exploratory, cell-by-cell version of the pipeline in `main.py`. Run `python ../src/make_synthetic_data.py` first if you don't have the real Kaggle dataset yet, or point `DATA_PATH` at `data/creditcard.csv` / your merged IEEE-CIS file.


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

from src.config import PipelineConfig
from src.data_loader import load_transactions
from src.preprocessing import split_data
from src.resampling import apply_smote, compute_scale_pos_weight
from src.model import train_xgboost
from src.threshold_tuning import tune_threshold
from src.evaluate import evaluate_on_test, plot_feature_importance, plot_threshold_sweep


## 0. IEEE-CIS only: merge transaction + identity tables

Skip this cell if you're using `creditcard.csv` (Option A in `data/README.md`) — it already has everything in one file.


In [ ]:
# transaction = pd.read_csv('../data/train_transaction.csv')
# identity = pd.read_csv('../data/train_identity.csv')
# merged = transaction.merge(identity, on='TransactionID', how='left')
# merged.to_csv('../data/ieee_merged.csv', index=False)


## 1. Load data

In [ ]:
DATA_PATH = '../data/synthetic_creditcard.csv'  # swap for the real file
TARGET_COL = 'Class'  # 'isFraud' for IEEE-CIS

df = load_transactions(DATA_PATH, TARGET_COL)
df.head()


## 2. EDA — how imbalanced are we, really?

In [ ]:
pos_rate = df[TARGET_COL].mean()
fig, ax = plt.subplots(figsize=(5, 4))
df[TARGET_COL].value_counts().plot(kind='bar', ax=ax)
ax.set_title(f'Class balance ({pos_rate:.3%} fraud)')
ax.set_xticklabels(['Not fraud', 'Fraud'], rotation=0)
plt.show()

print(df[TARGET_COL].value_counts())
print(f'Imbalance ratio: 1 fraud per {(1/pos_rate):.0f} transactions')


In [ ]:
if 'Amount' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.boxplot(x=TARGET_COL, y='Amount', data=df, ax=axes[0])
    axes[0].set_title('Amount by class (raw)')
    sns.boxplot(x=TARGET_COL, y=np.log1p(df['Amount']), data=df, ax=axes[1])
    axes[1].set_ylabel('log1p(Amount)')
    axes[1].set_title('Amount by class (log scale)')
    plt.tight_layout(); plt.show()


## 3. Split (before any resampling — see README 'Pitfalls avoided')

In [ ]:
cfg = PipelineConfig(data_path=DATA_PATH, target_col=TARGET_COL)
split = split_data(df, TARGET_COL, cfg.test_size, cfg.val_size, cfg.random_state)
split.X_train.shape, split.X_val.shape, split.X_test.shape


## 4. SMOTE oversampling (training fold only)

In [ ]:
X_train_res, y_train_res = apply_smote(
    split.X_train, split.y_train,
    sampling_strategy=cfg.smote_sampling_strategy,
    k_neighbors=cfg.smote_k_neighbors,
    random_state=cfg.random_state,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
split.y_train.value_counts().plot(kind='bar', ax=axes[0], title='Before SMOTE')
pd.Series(y_train_res).value_counts().plot(kind='bar', ax=axes[1], title='After SMOTE')
plt.tight_layout(); plt.show()


## 5. Train XGBoost (early stopping on validation PR-AUC)

In [ ]:
model = train_xgboost(X_train_res, y_train_res, split.X_val, split.y_val, cfg)


## 6. Threshold tuning on the validation set

In [ ]:
y_val_proba = model.predict_proba(split.X_val)[:, 1]
threshold_result = tune_threshold(split.y_val, y_val_proba, cfg)
plot_threshold_sweep(threshold_result.sweep, threshold_result.threshold, '../outputs')

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(threshold_result.sweep['threshold'], threshold_result.sweep['precision'], label='Precision')
ax.plot(threshold_result.sweep['threshold'], threshold_result.sweep['recall'], label='Recall')
ax.plot(threshold_result.sweep['threshold'], threshold_result.sweep['f1'], label='F1', linestyle='--')
ax.axvline(threshold_result.threshold, color='black', linestyle=':', label=f'chosen={threshold_result.threshold:.2f}')
ax.legend(); ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
plt.show()


## 7. Final evaluation on the held-out test set

Touched exactly once, after the threshold was already chosen — see README.


In [ ]:
metrics = evaluate_on_test(model, split.X_test, split.y_test, threshold_result.threshold, '../outputs')
metrics


## 8. Feature importance / interpretation

In [ ]:
plot_feature_importance(model, split.feature_names, '../outputs')
from PIL import Image
Image.open('../outputs/feature_importance.png')


**Reading gain vs. weight importance:** `gain` measures the average improvement in the loss function each time a feature is used to split; `weight` just counts how often it's used. They can disagree — a feature used rarely but decisively (high gain, low weight) vs. one used constantly for small refinements (high weight, low gain). Neither accounts for feature interactions the way SHAP values do; run with `--shap` (`main.py`) or add a `plot_shap_summary(...)` cell here if you want that level of rigor.


## 9. Compare against the scale_pos_weight baseline (optional)

Worth running once to see whether SMOTE actually earns its extra training cost on your dataset.


In [ ]:
spw = compute_scale_pos_weight(split.y_train)
baseline_model = train_xgboost(split.X_train, split.y_train, split.X_val, split.y_val, cfg, scale_pos_weight=spw)
baseline_val_proba = baseline_model.predict_proba(split.X_val)[:, 1]
from sklearn.metrics import average_precision_score
print('SMOTE val PR-AUC:      ', average_precision_score(split.y_val, y_val_proba))
print('scale_pos_weight val PR-AUC:', average_precision_score(split.y_val, baseline_val_proba))
